# Session 4: From Paper Trading to Production Operations

We have a validated Cobb-Douglas engine, EWLS-adapted SIM parameters, and paper trading positions in an Alpaca brokerage account. The question is no longer "does the strategy work?" — it's "can we operate it safely every day?" In this session, we build the production layer: a bandit selects which assets to trade, sentiment monitoring overrides risk appetite when markets deteriorate, and escalation triggers halt the system when safety limits are breached. Two AI layers meet in this session. 
* **The autonomous decision agent** is the Cobb-Douglas engine built across Sessions 2-3 (with its online EWLS SIM re-estimation and bandit-learned CES elasticity). It reads daily market data, updates its own parameters, selects allocations within a compliance envelope, and surfaces every trade to an audit log. 
* **Real-time AI monitoring** wraps the agent. A price-momentum sentiment classifier overrides risk appetite when markets deteriorate, and an optional LLM-based news-scoring pipeline reads corpus text via Claude and folds scores into preference weights as a third feature alongside SIM alpha and beta. 

The session treats the engine-plus-monitoring system as the operational AI pipeline any deployed decision system needs: continuous observation, automated escalation, explicit rollback, and post-hoc review.

> __Learning Objectives:__
>
> By the end of this session, you will be able to:
> * __Design a daily production loop on real positions:__ Combine bandit asset selection with Cobb-Douglas allocation on live Alpaca positions, using EWLS-adapted SIM parameters that update daily. Understand how the production loop differs from the historical replay in Session 3.
> * __Implement sentiment monitoring and escalation triggers:__ Compute real-time sentiment from price momentum and configure automatic lambda override when markets deteriorate. Define three escalation triggers (drawdown, sentiment crash, bandit churn) that operate on real brokerage account state.
> * __Build an operational monitoring dashboard:__ Construct a REST polling dashboard that tracks portfolio value, drawdown, and sentiment in near-real-time. Run what-if stress analysis to verify the escalation system responds correctly to hypothetical shocks.

## Examples
The following example notebooks accompany this lecture:

\u27a1 [Example: Production Portfolio Engine](eCornell-AI-Finance-S4-Example-ProductionPortfolioEngine-May-2026.ipynb). Run the full production loop on real Alpaca data: bandit asset selection, EWLS parameter updates, sentiment-driven lambda override, escalation triggers, and live order execution.

\u27a1 [Example: Operational Monitoring Dashboard](eCornell-AI-Finance-S4-Example-OperationalDashboard-May-2026.ipynb). Build a REST polling dashboard for near-real-time portfolio monitoring, track escalation triggers on live positions, and run what-if stress analysis on the current portfolio.

\ud83d\udcd8 [Deep Dive: Sentiment Signals and Escalation Thresholds](deepdive/eCornell-AI-Finance-S4-DeepDive-SentimentEscalation-May-2026.ipynb). Derivation of the sentiment signal from return momentum, analysis of escalation threshold sensitivity, and connection to portfolio insurance theory.

## From Paper Trading to Production: What Changes

In Session 3, we replayed history. The price matrix was fixed, every path was known before the engine started, and "trading" meant updating an array in memory. In production, one new row arrives each trading day — and the system must decide what to do before the next row exists.

> **1. Prices arrive one day at a time.** In Session 3, the price matrix had $T$ rows available from the start. In production, we fetch today's close from Alpaca and append it. Tomorrow's prices are unknown.

> **2. EWLS parameters update continuously.** The SIM parameters ($\alpha_i$, $\beta_i$, $\sigma_i$) are re-estimated each day as new observations arrive via exponentially weighted least squares. Yesterday's alpha may not be today's alpha.

> **3. Orders have real consequences.** Slippage, partial fills, and position drift between rebalances are production realities that backtesting ignores. The turnover cap becomes a hard operational constraint.

> **4. Three new layers sit on top of the engine.** The bandit selects which assets to trade. Sentiment monitoring adjusts risk appetite before losses materialize. Escalation triggers halt the system when safety limits are breached.

The daily production cadence runs as a numbered pipeline:

> 1. Fetch latest prices from Alpaca.
> 2. Update EWLS parameters with today's observation.
> 3. Compute sentiment from recent return momentum.
> 4. Determine effective $\lambda$ (with override if sentiment is below threshold).
> 5. Run bandit to select asset subset.
> 6. Check escalation triggers.
> 7. If safe: allocate via Cobb-Douglas, submit orders.
> 8. Log everything for audit trail.

This pipeline runs once per trading day. Each step depends on the one before it — sentiment must be computed before lambda is determined, and escalation must be checked before orders are submitted.

## Bandit Asset Selection on Live Data

In Session 3, the epsilon-greedy bandit operated on synthetic HMM paths. In production, it operates daily on real Alpaca data. The reward function uses EWLS-updated SIM parameters and live prices — so the reward landscape shifts each day as parameters adapt to new observations.

> **Reward computation.** At each production day $t$, the bandit evaluates an action vector $\mathbf{a} \in \{0,1\}^K$ by running the Cobb-Douglas allocator on the included assets and returning the resulting utility as the reward. Because the SIM parameters update daily via EWLS, the same action can yield different rewards on consecutive days.

> **Exploration decay.** The exploration rate $\epsilon_t = t^{-1/3} \cdot (K \cdot \ln t)^{1/3}$ decreases over production days. Early in deployment, the bandit explores aggressively. After weeks of operation, it mostly exploits its learned estimates.

> **Excluded assets.** Assets not selected by the bandit have their alpha set to $-10$ in the SIM parameter vector. This forces the Cobb-Douglas preference weight $\gamma_i$ to near-zero, effectively removing them from the allocation without changing the engine's structure.

The bandit and the Cobb-Douglas engine answer different questions:

| | Bandit Selector | Cobb-Douglas Engine |
|---|---|---|
| **Decides** | Which assets to include | How much of each asset |
| **Adapts via** | Trial-and-error learning | Sentiment signal $\lambda_t$ |
| **Strength** | Discovers non-obvious subsets | Fast, analytical, interpretable |
| **Weakness** | Needs iterations to converge | Always uses all included assets |

In Session 3, these two approaches competed. In production, they work together: the bandit decides _which_ assets, the Cobb-Douglas engine decides _how much_.

## Sentiment Monitoring with Lambda Override

The sentiment score measures market momentum and maps it to a signal the engine can act on.

> **Sentiment computation.** The sentiment score at time $t$ is computed from the market's recent return:
>
> $$s_t = \tanh\left(10 \cdot \frac{P_t - P_{t-5}}{P_{t-5}}\right)$$
>
> where $P_t$ is the market (SPY) close price. The $\tanh$ maps the scaled return to $[-1, 1]$: positive sentiment for rising markets, negative for falling.

> **Lambda override.** When $s_t < s_{\text{threshold}}$ (e.g., $s_{\text{threshold}} = -0.5$), the system overrides the EMA-based lambda with a defensive value $\lambda_{\text{override}}$ (e.g., 2.0). This increases risk aversion in the Cobb-Douglas preference weights, shifting allocation toward lower-beta assets before a drawdown fully materializes.

The override is _proactive_. It acts on momentum, a sustained price decline over the past five trading days, rather than waiting for portfolio equity to drop. The drawdown trigger (Section 4) is _reactive_: it fires after the loss has already occurred. Together they form a two-layer defense:

| Layer | Signal | Timing | Action |
|---|---|---|---|
| **Sentiment override** | 5-day return momentum | Before losses materialize | Increase $\lambda$, shift to defensive allocation |
| **Drawdown trigger** | Portfolio equity vs. peak | After losses materialize | Halt trading, move to 100% cash |

The sentiment override reduces the probability that the drawdown trigger ever fires. When it does fire, the portfolio has already been partially de-risked.

> __Same score, different mechanism:__
>
> Our sentiment score $s_t$ reports a number, not a reason. A sustained five-day decline can arise from a slow volatility drift inside a bear regime, from a sharp one-day news shock that drags the trailing window, or from a scheduled macro print the market had already priced before our filter could react. All three produce similar $s_t$ values; all three imply different forward dynamics. __Prof. Chau's lecture in module 1 of this course__ makes the broader version of this point at the language level: prediction tells us _that_, inference tells us _why_, and a monitoring pipeline that acts only on _that_ is asking the operator to supply the _why_ in real time. The engine's contract on our book is explicit about this split. A sentiment reading below threshold triggers the defensive lambda automatically; the investment committee owns the diagnosis of which mechanism produced the reading and whether the automatic response is the right one.

## Escalation Triggers on Real Positions

The production system defines three escalation triggers. Each is computed from real Alpaca account state — not simulated values.

> **1. Drawdown (Critical).** If portfolio equity from `get_account` has declined more than $d_{\max}$ from its peak:
>
> $$(W_{\text{peak}} - W_t) / W_{\text{peak}} > d_{\max}$$
>
> Action: de-risk to 100% cash. Notify investment committee.

> **2. Sentiment Crash (Warning).** If $s_t < s_{\text{threshold}}$:
>
> Action: override lambda to defensive value. Log for review. Continue trading.

> **3. Bandit Churn (Warning).** If the bandit changes more than $C_{\max}$ assets in one day:
>
> $$\sum_{i=1}^{K} |a_t^{(i)} - a_{t-1}^{(i)}| > C_{\max}$$
>
> Action: hold previous allocation. The bandit may be oscillating — review its state.

The severity hierarchy determines the response. Warnings modify behavior: sentiment crash increases risk aversion, bandit churn freezes the allocation. Critical triggers halt trading entirely and liquidate to cash. Both warning and critical events are logged with timestamps, trigger values, and the system's response for investment committee review.

| Severity | Effect | Examples |
|---|---|---|
| **Warning** | Modifies behavior, continues operating | Sentiment crash, bandit churn |
| **Critical** | Halts trading, moves to cash | Drawdown breach |

## Operational Monitoring

The monitoring dashboard polls the Alpaca API and checks the safety system continuously during market hours.

> **REST polling pattern.** The dashboard fetches account state, positions, and quotes every 30 seconds. On each poll it computes running metrics: P&L from session open, drawdown from session peak, and sentiment from the latest SPY price. All three escalation triggers are checked on each poll cycle.

> **Audit timeline.** Every poll cycle writes a timestamped record: portfolio value, drawdown percentage, sentiment score, trigger distances (how close each trigger is to firing), and any actions taken. This timeline answers the question: _when did the system approach its limits, and what did it do?_

### What-If Stress Analysis

The dashboard also supports hypothetical shock analysis on the current portfolio.

> **Shock application.** Apply hypothetical market drops ($-10\%$, $-20\%$, $-30\%$) to current position prices. For each shock level:
>
> 1. Recompute portfolio value at shocked prices.
> 2. Compute the implied drawdown from the recorded peak.
> 3. Check which escalation triggers would fire.
> 4. Report the result: "if the market drops 20% right now, does the system protect capital?"

This is the operational equivalent of the Monte Carlo backtest from Session 3. The difference is that the positions are real. A stress test that shows the drawdown trigger would not fire at $-20\%$ means the threshold needs tightening — before the drop happens, not after.

## The Arc of a Profitable Strategy

The monitoring dashboard answers what is happening today. It does not answer what happens when the same strategy ships from a dozen firms at the same time.

> __Discovery, Adoption, Crowding, Compression:__
>
> __Prof. Chau's lecture in module 1 of this course__ traces this arc through two prior waves of quantitative edge. Statistical arbitrage produced Sharpe ratios above 2.0 at D.E. Shaw and Renaissance in the early 1990s; AUM in quant equity grew from roughly \$100B to \$500B between 2000 and 2007; August 2007 unwound \$1.4T of correlated positions in days. High-frequency trading followed the same path: genuine spread compression for investors in the 2000s, with the firms running the strategy watching their own margins erode as competition intensified. Lopez-Lira and Tang (2023) document long-positive / short-negative LLM sentiment signals earning large out-of-sample returns, a Discovery-stage result, with Adoption and Crowding yet to be tested empirically. The pattern that kills a strategy is not prediction error; it is competitors adopting the same signal.

Two consequences land on the production layer. First, the metric that matters for long-run capacity is not Sharpe alone but _Sharpe against crowding_: what does our realized edge look like when a meaningful fraction of the float is run by allocators making the same trade? Our current dashboard does not measure this because on synthetic data it cannot change. A live version of the question would track rolling out-of-sample Sharpe against a historical baseline and flag persistent compression, not a single bad week.

Second, the trigger logic should assume the edge is finite. The drawdown gate, the turnover cap, and the bandit churn limit are all designed to survive the period when the strategy works. They also need to degrade gracefully through the period when it stops. The right monitoring question at end-of-day is not "did we beat the benchmark today?" but "is our realized edge on the same distribution it was six months ago?" When the answer is no, the response is a committee review, not a parameter tweak.

## Summary

> __Key Takeaways:__
>
> * __The production loop adds decision layers:__ The bandit selects assets, sentiment adjusts risk appetite, and escalation triggers enforce safety limits — all operating on real brokerage data via the Alpaca API. Each layer can be tuned independently without changing the underlying Cobb-Douglas allocation engine.
> * __Sentiment monitoring is proactive, escalation is reactive:__ The sentiment override shifts allocation before losses materialize, while drawdown triggers halt the system after the fact. Together they form a two-layer defense: reduce risk early, then stop trading if reduction was not enough.
> * __What-if stress analysis validates the safety system:__ Applying hypothetical shocks to live positions and checking whether escalation triggers fire correctly is the operational equivalent of the Monte Carlo backtest from Session 3. The difference is that the positions are real and the consequences of failure are immediate.

This is the final session of the course. The full pipeline — from SIM estimation (Session 1) through adaptive allocation (Session 2), regime-aware validation (Session 3), to live production operations (Session 4) — is now complete.

### Disclaimer
This content is for educational purposes only and does not constitute investment advice. The production system described is a pedagogical tool using synthetic data and paper trading. Real-world deployment requires regulatory approval, compliance review, operational infrastructure, and risk management beyond the scope of this course.